In [26]:
#IMPORTAÇÕES
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error
from statsmodels.tools.eval_measures import rmse
from datetime import datetime
import os

In [27]:
print("=== PROCESSANDO DADOS DO EXCEL ===")
# Lendo os dados do Excel (código original adaptado)
df_excel = pd.read_excel(
    "../Controle_Faturamento_2023.xlsx",
    sheet_name="Consumo Médio - Clientes",
)

# Removendo dados desnecessários (mantendo UC)
df_excel_clean = df_excel.copy()
df_excel_clean.drop(df_excel_clean.iloc[:, 48:68], inplace=True, axis=1)
df_excel_clean.drop("Nº Cliente", inplace=True, axis=1)
df_excel_clean.drop("Cliente", inplace=True, axis=1)

df_excel_clean = df_excel_clean.head()

=== PROCESSANDO DADOS DO EXCEL ===


In [28]:
# Extraindo os UCs que serão usados para filtrar os dados
selected_ucs = df_excel_clean['UC'].astype(str).tolist()
print(f"UCs selecionados do Excel: {selected_ucs}")

# Corrigindo os dados para uso das fórmulas (Colocando a data como índice do dataset)
df_excel_transpose1 = df_excel_clean.transpose()
df_excel_transpose = df_excel_transpose1.rename(columns=df_excel_transpose1.iloc[0])
df_excel_transpose.drop("UC", inplace=True, axis=0)
df_excel_transpose.dropna(inplace=True)

UCs selecionados do Excel: ['3003858507', '3001084033', '3011373971', '3001449459', '3011504476']


In [29]:
# Transformando índice em datetime para o Excel
some_dates = np.array(df_excel_transpose.index, dtype="datetime64[D]")
idx_excel = pd.DatetimeIndex(some_dates)
df_excel_transpose.index = idx_excel

# Convertendo Excel para formato long (igual ao CSV)
excel_long = []
for installation in df_excel_transpose.columns:
    for date in df_excel_transpose.index:
        value = df_excel_transpose.loc[date, installation]
        if pd.notna(value):  # Só adiciona valores não-nulos
            excel_long.append({
                'Date': date,
                'Installation': str(installation),  # Convertendo para string para compatibilidade
                'ConsumeValue': float(value),
                'Source': 'Excel'
            })

df_excel_long = pd.DataFrame(excel_long)

In [30]:
# Lendo os dados do CSV
df_csv = pd.read_csv("../_SELECT_cb_Name_cb_Installation_fc_ForecastConsumeValue_fc_Refer_202504140945.csv")

# Convertendo a coluna Date para datetime
df_csv['Date'] = pd.to_datetime(df_csv['Date'])
df_csv['Installation'] = df_csv['Installation'].astype(str)  # Garantindo que seja string

In [31]:
# Filtrando CSV para manter apenas os mesmos UCs do Excel
df_csv_filtered = df_csv[df_csv['Installation'].isin(selected_ucs)].copy()
df_csv_filtered['Source'] = 'CSV'

In [32]:
print("\n=== COMBINANDO OS DADOS ===")
# Combinando os dataframes
df_combined = pd.concat([df_excel_long, df_csv_filtered], ignore_index=True)

# Verificando duplicatas na combinação Date + Installation
duplicate_check = df_combined.groupby(['Date', 'Installation']).size()
duplicates = duplicate_check[duplicate_check > 1]

if len(duplicates) > 0:
    print(f"\nEncontradas {len(duplicates)} combinações Date-Installation duplicadas")
    print("Resolvendo duplicatas...")
    
    df_combined_clean = df_combined.sort_values(['Date', 'Installation', 'Source']).drop_duplicates(
        subset=['Date', 'Installation'], keep='last')  # CSV vem depois do Excel na ordenação
    
    print(f"Dados após resolver duplicatas: {len(df_combined_clean)} registros")
else:
    df_combined_clean = df_combined.copy()
    print("Nenhuma duplicata encontrada!")


=== COMBINANDO OS DADOS ===

Encontradas 10 combinações Date-Installation duplicadas
Resolvendo duplicatas...
Dados após resolver duplicatas: 354 registros


In [33]:
# Fazendo o pivot com os dados limpos
df_final = df_combined_clean.pivot(index='Date', columns='Installation', values='ConsumeValue')

# Substituindo valores NaN por 0
df_final.fillna(0, inplace=True)

# Ordenando o índice por data
df_final.sort_index(inplace=True)

# Removendo dados a partir de 2025-01-01
df_final = df_final[df_final.index < '2025-01-01']

In [34]:
# Se a frequência não for detectada ou for diferente de mensal, resample para mensal
date_diffs = df_final.index.to_series().diff().dropna()
print(f"Diferenças entre datas: {date_diffs.value_counts().head()}")

# Se os dados já são mensais, tenta inferir a frequência
try:
    df_final.index.freq = pd.infer_freq(df_final.index)
    print(f"Frequência inferida: {df_final.index.freq}")
except:
    print("Não foi possível inferir frequência automaticamente")

# Se a frequência não for detectada ou for diferente de mensal, resample para mensal
if df_final.index.freq != 'MS' and df_final.index.freq != 'M':
    print("Reamostrando dados para frequência mensal...")
    # Agrupa por mês e soma os valores (ou use .mean() se preferir média)
    df_final = df_final.resample('MS').mean()
    df_final.index.freq = 'MS'
    print(f"Nova frequência: {df_final.index.freq}")
    print(f"Novo shape após reamostragem: {df_final.shape}")
else:
    print("Dados já estão em frequência mensal")

Diferenças entre datas: Date
31 days    35
30 days    20
28 days     5
29 days     2
2 days      2
Name: count, dtype: int64
Frequência inferida: None
Reamostrando dados para frequência mensal...
Nova frequência: <MonthBegin>
Novo shape após reamostragem: (67, 5)


In [35]:
print("\n=== ESTRUTURA FINAL DOS DADOS ===")
print("Colunas disponíveis (Installations):")
print(df_final.columns.tolist())
print(f"\nShape dos dados: {df_final.shape}")
print(f"Período dos dados: {df_final.index.min()} até {df_final.index.max()}")


=== ESTRUTURA FINAL DOS DADOS ===
Colunas disponíveis (Installations):
['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']

Shape dos dados: (67, 5)
Período dos dados: 2019-06-01 00:00:00 até 2024-12-01 00:00:00


In [36]:
# ============================================================================
# PROCESSAMENTO SEQUENCIAL DE TODAS AS INSTALAÇÕES
# ============================================================================

# Lista de todas as instalações para processar
INSTALLATIONS = ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']

# Definindo os ranges dos hiperparâmetros
yearly_seasonality_range = range(0, 13)  # 0-12
total_combinations = len(yearly_seasonality_range)

print(f"\n{'='*80}")
print(f"PROPHET GRID SEARCH SEQUENCIAL - TODAS AS INSTALAÇÕES")
print(f"{'='*80}")
print(f"Instalações a processar: {INSTALLATIONS}")
print(f"Combinações por instalação: {total_combinations}")
print(f"Total geral: {len(INSTALLATIONS) * total_combinations}")
print(f"{'='*80}\n")

# Função para verificar se uma combinação já foi testada
def already_tested(existing_results, yearly_seasonality):
    if len(existing_results) == 0:
        return False
    mask = (existing_results['yearly_seasonality'] == yearly_seasonality)
    return mask.any()

# Função para salvar resultado
def save_result(results_file, installation_id, yearly_seasonality, mape, mae, wmape, rmse_val, status, error_msg=""):
    result = {
        'Installation': installation_id,
        'yearly_seasonality': yearly_seasonality,
        'MAPE': mape,
        'MAE': mae,
        'WMAPE': wmape,
        'RMSE': rmse_val,
        'Status': status,
        'Error_Message': error_msg,
        'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    result_df = pd.DataFrame([result])
    result_df.to_csv(results_file, mode='a', header=False, index=False)

# Loop principal para todas as instalações
overall_start_time = datetime.now()
total_processed = 0
total_success = 0
total_errors = 0

for installation_idx, INSTALLATION_ID in enumerate(INSTALLATIONS):
    
    print(f"\n{'='*80}")
    print(f"PROCESSANDO INSTALAÇÃO {installation_idx + 1}/{len(INSTALLATIONS)}: {INSTALLATION_ID}")
    print(f"{'='*80}")
    
    # Verifica se a coluna existe
    if INSTALLATION_ID not in df_final.columns:
        print(f"ERRO: Installation {INSTALLATION_ID} não encontrada no dataframe!")
        print(f"Colunas disponíveis: {df_final.columns.tolist()}")
        continue
    
    # Nome do arquivo de resultados
    results_file = f"prophet_results_{INSTALLATION_ID}.csv"
    
    # Inicializa ou carrega arquivo existente
    if os.path.exists(results_file):
        print(f"Arquivo {results_file} encontrado.")
        existing_results = pd.read_csv(results_file)
        print(f"Resultados anteriores: {len(existing_results)} registros")
    else:
        print(f"Criando novo arquivo {results_file}...")
        existing_results = pd.DataFrame(columns=[
            'Installation', 'yearly_seasonality',
            'MAPE', 'MAE', 'WMAPE', 'RMSE', 
            'Status', 'Error_Message', 'Timestamp'
        ])
        existing_results.to_csv(results_file, index=False)
        print("Arquivo criado!")
    
    # Prepara os dados para Prophet (igual ao SARIMA)
    print("Preparando dados...")
    data = df_final[INSTALLATION_ID]
    end = len(data) - 1
    half = data.iloc[23:]  # Mesmo split do SARIMA
    
    # Converte para formato Prophet - renomeia todas as colunas
    prophet_data = df_final.reset_index()
    
    # Renomeia as colunas: primeira é 'ds', outras são 'y1', 'y2', etc.
    new_columns = ['ds'] + [f'y{i+1}' for i in range(len(df_final.columns))]
    prophet_data.columns = new_columns
    
    # Encontra qual coluna corresponde à instalação atual e renomeia para 'y'
    installation_idx = list(df_final.columns).index(INSTALLATION_ID)
    target_column = f'y{installation_idx + 1}'
    
    # Renomeia a coluna target para 'y'
    column_mapping = {target_column: 'y'}
    prophet_data = prophet_data.rename(columns=column_mapping)
    
    print(f"Colunas após renomeação: {prophet_data.columns.tolist()}")
    print(f"Usando coluna 'y' (era {target_column}, instalação {INSTALLATION_ID})")
    
    print(f"Dados preparados: {len(prophet_data)} registros")
    print(f"Período: {prophet_data['ds'].min()} até {prophet_data['ds'].max()}")
    
    # Contadores para esta instalação
    combination_count = 0
    skipped_count = 0
    success_count = 0
    error_count = 0
    
    start_time = datetime.now()
    results_buffer = []
    
    print(f"\nInício do processamento: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("-" * 80)
    
    # Loop para yearly_seasonality
    for yearly_seasonality in yearly_seasonality_range:
        combination_count += 1
        
        # Verifica se já foi testado
        if already_tested(existing_results, yearly_seasonality):
            skipped_count += 1
            print(f"yearly_seasonality={yearly_seasonality:2d} - PULADO (já testado)")
            continue
        
        try:
            # Cria e treina o modelo Prophet
            m = Prophet(growth='flat', yearly_seasonality=yearly_seasonality)
            m.fit(prophet_data[['ds', 'y']])  # Usa apenas ds e y para o fit
            
            # Faz predições para todo o período
            future = m.make_future_dataframe(periods=0)
            forecast = m.predict(future)
            
            # Alinha predições com dados originais (usando os índices originais)
            predictions = pd.Series(forecast['yhat'].values, index=data.index)
            half_pred = predictions.iloc[23:]  # Mesmo split do SARIMA
            
            # Calcula as métricas (mesmas do SARIMA)
            mape = mean_absolute_percentage_error(half, half_pred)
            mae = mean_absolute_error(half, half_pred)
            wmape = mean_absolute_percentage_error(half, half_pred, sample_weight=half)
            rmse_val = rmse(half, half_pred)
            
            # Adiciona ao buffer
            results_buffer.append({
                'Installation': INSTALLATION_ID,
                'yearly_seasonality': yearly_seasonality,
                'MAPE': mape,
                'MAE': mae,
                'WMAPE': wmape,
                'RMSE': rmse_val,
                'Status': "Success",
                'Error_Message': "",
                'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            success_count += 1
            print(f"yearly_seasonality={yearly_seasonality:2d} - SUCESSO | MAPE: {mape*100:6.2f}% | MAE: {mae:8.2f} | RMSE: {rmse_val:8.2f}")
            
        except Exception as e:
            error_count += 1
            error_msg = str(e)[:200]
            
            print(f"yearly_seasonality={yearly_seasonality:2d} - ERRO: {error_msg[:50]}...")
            
            # Adiciona ao buffer
            results_buffer.append({
                'Installation': INSTALLATION_ID,
                'yearly_seasonality': yearly_seasonality,
                'MAPE': np.nan,
                'MAE': np.nan,
                'WMAPE': np.nan,
                'RMSE': np.nan,
                'Status': "Error",
                'Error_Message': error_msg,
                'Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
    
    # Salva todos os resultados desta instalação
    if results_buffer:
        try:
            buffer_df = pd.DataFrame(results_buffer)
            buffer_df.to_csv(results_file, mode='a', header=False, index=False)
            print(f"\nSalvou {len(results_buffer)} resultados para {INSTALLATION_ID}")
        except Exception as e:
            print(f"ERRO ao salvar resultados para {INSTALLATION_ID}: {e}")
    
    # Resumo desta instalação
    end_time = datetime.now()
    elapsed = (end_time - start_time).total_seconds()
    
    print(f"\n{'-'*80}")
    print(f"INSTALAÇÃO {INSTALLATION_ID} CONCLUÍDA")
    print(f"{'-'*80}")
    print(f"Tempo: {elapsed:.1f}s | Testados: {combination_count} | Pulados: {skipped_count}")
    print(f"Sucessos: {success_count} | Erros: {error_count}")
    
    # Mostra o melhor resultado desta instalação
    final_results = pd.read_csv(results_file)
    successful = final_results[final_results['Status'] == 'Success'].copy()
    
    if len(successful) > 0:
        successful = successful.sort_values('MAPE')
        best = successful.iloc[0]
        
        print(f"\nMELHOR MODELO para {INSTALLATION_ID}:")
        print(f"yearly_seasonality={int(best['yearly_seasonality'])} | MAPE: {best['MAPE']*100:.2f}% | RMSE: {best['RMSE']:.2f}")
    else:
        print(f"\nNenhum teste bem-sucedido para {INSTALLATION_ID}")
    
    # Atualiza contadores globais
    total_processed += combination_count - skipped_count
    total_success += success_count
    total_errors += error_count

# Resumo final de todas as instalações
overall_end_time = datetime.now()
overall_elapsed = (overall_end_time - overall_start_time).total_seconds()

print(f"\n\n{'='*80}")
print(f"PROCESSAMENTO COMPLETO - TODAS AS INSTALAÇÕES")
print(f"{'='*80}")
print(f"Tempo total: {overall_elapsed/60:.2f} minutos")
print(f"Instalações processadas: {len(INSTALLATIONS)}")
print(f"Combinações processadas: {total_processed}")
print(f"Total de sucessos: {total_success}")
print(f"Total de erros: {total_errors}")
print(f"{'='*80}")

# Resumo consolidado dos melhores modelos
print(f"\nRESUMO DOS MELHORES MODELOS:")
print(f"{'='*80}")
print(f"{'Installation':<12} {'Best_YS':<8} {'MAPE':<8} {'MAE':<10} {'RMSE':<10} {'Status'}")
print(f"{'-'*80}")

for installation_id in INSTALLATIONS:
    results_file = f"prophet_results_{installation_id}.csv"
    if os.path.exists(results_file):
        try:
            results = pd.read_csv(results_file)
            successful = results[results['Status'] == 'Success']
            
            if len(successful) > 0:
                best = successful.sort_values('MAPE').iloc[0]
                print(f"{installation_id:<12} {int(best['yearly_seasonality']):<8} {best['MAPE']*100:6.2f}% {best['MAE']:8.2f} {best['RMSE']:8.2f}   OK")
            else:
                print(f"{installation_id:<12} {'N/A':<8} {'N/A':<8} {'N/A':<10} {'N/A':<10}   ERROR")
        except Exception as e:
            print(f"{installation_id:<12} {'N/A':<8} {'N/A':<8} {'N/A':<10} {'N/A':<10}   FILE_ERROR")
    else:
        print(f"{installation_id:<12} {'N/A':<8} {'N/A':<8} {'N/A':<10} {'N/A':<10}   NO_FILE")

print(f"{'='*80}")
print(f"Arquivos gerados: prophet_results_[INSTALLATION_ID].csv")
print(f"{'='*80}")


PROPHET GRID SEARCH SEQUENCIAL - TODAS AS INSTALAÇÕES
Instalações a processar: ['3001084033', '3001449459', '3003858507', '3011373971', '3011504476']
Combinações por instalação: 13
Total geral: 65


PROCESSANDO INSTALAÇÃO 1/5: 3001084033
Arquivo prophet_results_3001084033.csv encontrado.
Resultados anteriores: 13 registros
Preparando dados...
Colunas após renomeação: ['ds', 'y', 'y2', 'y3', 'y4', 'y5']
Usando coluna 'y' (era y1, instalação 3001084033)
Dados preparados: 67 registros
Período: 2019-06-01 00:00:00 até 2024-12-01 00:00:00

Início do processamento: 2025-10-19 14:42:04
--------------------------------------------------------------------------------
yearly_seasonality= 0 - PULADO (já testado)
yearly_seasonality= 1 - PULADO (já testado)
yearly_seasonality= 2 - PULADO (já testado)
yearly_seasonality= 3 - PULADO (já testado)
yearly_seasonality= 4 - PULADO (já testado)
yearly_seasonality= 5 - PULADO (já testado)
yearly_seasonality= 6 - PULADO (já testado)
yearly_seasonality= 7 - 

14:42:04 - cmdstanpy - INFO - Chain [1] start processing
14:42:04 - cmdstanpy - INFO - Chain [1] done processing


yearly_seasonality= 0 - SUCESSO | MAPE: 10727492428766930944.00% | MAE:   398.75 | RMSE:   493.52


14:42:04 - cmdstanpy - INFO - Chain [1] start processing
14:42:04 - cmdstanpy - INFO - Chain [1] done processing


yearly_seasonality= 1 - SUCESSO | MAPE: 9737784140896894976.00% | MAE:   404.14 | RMSE:   487.99


...

yearly_seasonality= 8 - SUCESSO | MAPE: 40868206186284138496.00% | MAE:   607.57 | RMSE:   744.67
yearly_seasonality= 9 - SUCESSO | MAPE: 41989173239250870272.00% | MAE:   602.36 | RMSE:   735.43


14:42:11 - cmdstanpy - INFO - Chain [1] start processing
14:42:11 - cmdstanpy - INFO - Chain [1] done processing
14:42:11 - cmdstanpy - INFO - Chain [1] start processing
14:42:11 - cmdstanpy - INFO - Chain [1] done processing


yearly_seasonality=10 - SUCESSO | MAPE: 39914512824064147456.00% | MAE:   577.15 | RMSE:   699.63
yearly_seasonality=11 - SUCESSO | MAPE: 33107083816320581632.00% | MAE:   556.01 | RMSE:   673.25


14:42:11 - cmdstanpy - INFO - Chain [1] start processing
14:42:11 - cmdstanpy - INFO - Chain [1] done processing


yearly_seasonality=12 - SUCESSO | MAPE: 28576990395380289536.00% | MAE:   546.25 | RMSE:   641.54

Salvou 13 resultados para 3011504476

--------------------------------------------------------------------------------
INSTALAÇÃO 3011504476 CONCLUÍDA
--------------------------------------------------------------------------------
Tempo: 2.5s | Testados: 13 | Pulados: 0
Sucessos: 13 | Erros: 0

MELHOR MODELO para 3011504476:
yearly_seasonality=12 | MAPE: 28576990395380289536.00% | RMSE: 641.54


PROCESSAMENTO COMPLETO - TODAS AS INSTALAÇÕES
Tempo total: 0.11 minutos
Instalações processadas: 5
Combinações processadas: 39
Total de sucessos: 39
Total de erros: 0

RESUMO DOS MELHORES MODELOS:
Installation Best_YS  MAPE     MAE        RMSE       Status
--------------------------------------------------------------------------------
3001084033   12        13.73%   708.05   929.10   OK
3001449459   12       6697820881001806848.00%   301.13   336.92   OK
3003858507   12        39.34%   604.24   